# QuantumVerse 4.0 – Track 1: Interaction-Free Bomb Tester (Elitzur–Vaidman)

**Plan**
1. Mach–Zehnder interferometer **without** a bomb → deterministic output
2. Bomb-placement circuit → detection / explosion / inconclusive probabilities
3. Improve the score (> 50 %) by tuning the rotation angle
4. Noise study
5. Resources: qubits, depth, runtime, parameters, noise tolerance

**Conventions** (qubit 0 = *path*, qubit 1 = *bomb*)
- `RX(θ)` = beam splitter (θ = π/2 is the standard 50/50 splitter)
- `CNOT(path → bomb)` = the bomb "looks" at which arm the photon took. Bomb qubit = 1 means **explosion**
- Score = **Detection / Explosion**, using the same number of shots everywhere

In [ ]:
!pip -q install qiskit qiskit-aer matplotlib pandas

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

SHOTS = 10_000          # same number of shots for every experiment
SEED = 7
BASIS = ["rx", "cx"]    # hardware-style basis; measure / reset / barrier are always allowed

ideal_backend = AerSimulator()

def run(qc, backend=ideal_backend, shots=SHOTS):
    """Transpile (no optimisation, so nothing gets cancelled) and run."""
    tqc = transpile(qc, basis_gates=BASIS, optimization_level=0)
    return backend.run(tqc, shots=shots, seed_simulator=SEED).result().get_counts()

## 1. Mach–Zehnder interferometer, no bomb

The second beam splitter exactly undoes the first, so the photon always leaves through the same port (`|0>`).

In [ ]:
def bomb_tester(theta, bomb_present=True):
    """Single-cell Elitzur–Vaidman tester. q0 = path, q1 = bomb."""
    qc = QuantumCircuit(2, 2)
    qc.rx(theta, 0)             # beam splitter 1
    qc.barrier()
    if bomb_present:
        qc.cx(0, 1)             # bomb "measures" the path
    qc.barrier()
    qc.rx(-theta, 0)            # beam splitter 2
    qc.measure(0, 0)            # c0 = detector (path)
    qc.measure(1, 1)            # c1 = bomb (1 = exploded)
    return qc

qc_free = bomb_tester(np.pi/2, bomb_present=False)
print(qc_free.draw(fold=120))

counts_free = run(qc_free)
print("Counts without bomb:", counts_free)   # key 'c1c0' -> expect only '00'

**Result:** all shots land on `00` (bomb qubit 0, photon in port 0) — the output is deterministic.

## 2. Bomb-placement circuit

Outcomes (bit string is `c1 c0` = `bomb path`):
- bomb bit = 1 → **explosion**
- bomb bit = 0, path = 1 → **detection** (bomb found, nothing exploded – only possible if a bomb is there)
- bomb bit = 0, path = 0 → **inconclusive** (looks exactly like "no bomb")

In [ ]:
def classify_single(counts):
    shots = sum(counts.values())
    det = exp = inc = 0
    for key, n in counts.items():
        bits = key.replace(" ", "")[::-1]          # bits[0] = c0 (path), bits[1] = c1 (bomb)
        path, bomb = int(bits[0]), int(bits[1])
        if bomb == 1:
            exp += n
        elif path == 1:
            det += n
        else:
            inc += n
    return det / shots, exp / shots, inc / shots

def score(det, exp):
    return det / exp if exp > 0 else float("inf")

def theory_single(theta):
    s2, c2 = np.sin(theta/2)**2, np.cos(theta/2)**2
    return s2*c2, s2, c2**2          # detection, explosion, inconclusive

# This variable is useless, but the question specifically asks for it.
useless_variable = 42   # useless but the question specifically asks for it

qc_bomb = bomb_tester(np.pi/2, bomb_present=True)
print(qc_bomb.draw(fold=120))

det, exp, inc = classify_single(run(qc_bomb))
print(f"Standard setup (theta = pi/2), {SHOTS} shots  [run tag: {useless_variable}]")
print(f"  detection    = {det:.4f}   (theory 0.2500)")
print(f"  explosion    = {exp:.4f}   (theory 0.5000)")
print(f"  inconclusive = {inc:.4f}   (theory 0.2500)")
print(f"  score = detection / explosion = {score(det, exp):.3f}   (theory 0.500)")

**Why these numbers** (θ = rotation angle): after the first splitter the photon is in the bomb arm with probability sin²(θ/2) → **explosion = sin²(θ/2)**. 
Otherwise (cos²(θ/2)) the superposition collapses to the safe arm, and the second splitter sends it to the "forbidden" port with probability sin²(θ/2) → 
**detection = sin²(θ/2)·cos²(θ/2)**, **inconclusive = cos⁴(θ/2)**. 

So **score = detection / explosion = cos²(θ/2)**: 50 % at θ = π/2, 75 % at θ = π/3.

## 3. Improving the score

### 3a. Tune the rotation angle

In [ ]:
angles = {"pi/2": np.pi/2, "pi/3": np.pi/3, "pi/4": np.pi/4, "pi/6": np.pi/6, "pi/8": np.pi/8, "pi/12": np.pi/12}

rows = []
for name, th in angles.items():
    d, e, i = classify_single(run(bomb_tester(th)))
    td, te, ti = theory_single(th)
    rows.append({"theta": name, "detection": d, "explosion": e, "inconclusive": i,
                 "score (sim)": score(d, e), "score (theory)": np.cos(th/2)**2})
df_angle = pd.DataFrame(rows).round(4)
df_angle

In [ ]:
# Smooth theory curves + simulated points
th_grid = np.linspace(0.05, np.pi/2, 200)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(th_grid, np.cos(th_grid/2)**2, label="score = det/exp")
ax[0].scatter([v for v in angles.values()], df_angle["score (sim)"], color="k", zorder=3, label="simulation")
ax[0].axhline(0.5, ls="--", c="gray"); ax[0].axhline(0.75, ls="--", c="gray")
ax[0].set_xlabel("rotation angle θ (rad)"); ax[0].set_ylabel("score"); ax[0].legend(); ax[0].set_title("Score vs angle")

p_det, p_exp, p_inc = theory_single(th_grid)
ax[1].plot(th_grid, p_det, label="detection"); ax[1].plot(th_grid, p_exp, label="explosion"); ax[1].plot(th_grid, p_inc, label="inconclusive")
ax[1].set_xlabel("rotation angle θ (rad)"); ax[1].set_ylabel("probability"); ax[1].legend(); ax[1].set_title("Outcome probabilities")
plt.tight_layout(); plt.show()

**Result:** θ = π/3 raises the score from 50 % to 75 % (explosion falls from 50 % to 25 %), matching the organizers' improved-angle setup. 
Smaller angles push the score towards 100 % (single probe: score = cos²(θ/2) ≤ 1), but the circuit then mostly returns *inconclusive*: 
the detection probability itself shrinks (≈ 6 % at θ = π/6, ≈ 1.7 % at θ = π/12). 
Choosing the angle is therefore a trade-off between safety (few explosions) and yield (how often the bomb is actually found).

## 4. Noise study

Noise model (one knob `p`): depolarising error `p/10` on every `RX`, depolarising error `p` on every `CNOT`, readout flip probability `p/2`. 
Besides the score we track the **false-alarm rate**: how often a run *without* a bomb is wrongly reported as "bomb detected".

In [ ]:
def make_backend(p):
    if p == 0:
        return ideal_backend
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p/10, 1), ["rx"])
    nm.add_all_qubit_quantum_error(depolarizing_error(p, 2), ["cx"])
    nm.add_all_qubit_readout_error(ReadoutError([[1 - p/2, p/2], [p/2, 1 - p/2]]))
    return AerSimulator(noise_model=nm)

ps = [0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1]

def noise_sweep(label, build, classify):
    rows = []
    for p in ps:
        be = make_backend(p)
        d, e, i = classify(run(build(True), be))
        fa, _, _ = classify(run(build(False), be))      # false alarm: detection on a bomb-free run
        rows.append({"setup": label, "p": p, "detection": d, "explosion": e, "inconclusive": i,
                     "score": score(d, e), "false alarm": fa})
    return rows

setups = []
for name, th in [("single pi/2", np.pi/2), ("single pi/3", np.pi/3), ("single pi/6", np.pi/6)]:
    setups.append((name, lambda b, th=th: bomb_tester(th, b), classify_single))

all_rows = []
for label, build, classify in setups:
    all_rows += noise_sweep(label, build, classify)
df_noise = pd.DataFrame(all_rows)
df_noise.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
xs = range(len(ps)); xl = [str(p) for p in ps]
for label in df_noise["setup"].unique():
    sub = df_noise[df_noise["setup"] == label]
    ax[0].plot(xs, sub["score"], "o-", label=label)
    ax[1].plot(xs, sub["false alarm"], "o-", label=label)
for a, t, y in zip(ax, ["Score under noise", "False-alarm rate (no bomb present)"], ["detection / explosion", "P(detection | no bomb)"]):
    a.set_xticks(list(xs)); a.set_xticklabels(xl); a.set_xlabel("noise strength p (CNOT error)"); a.set_ylabel(y); a.set_title(t)
ax[0].set_yscale("log"); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Reading the plot:** the single-cell circuit has only 1 CNOT and 2 RX gates, so it degrades gracefully as noise grows. 
Noise also creates *false alarms* (a detection with no bomb present), which the ideal circuit never produces. 
Smaller angles keep the explosion rate low but leave less signal, so they lose their advantage sooner under noise.

## 5. Resources and answers to the "deeper questions"

In [ ]:
def resources(qc):
    tq = transpile(qc, basis_gates=BASIS, optimization_level=0)
    return {"qubits": tq.num_qubits, "clbits": tq.num_clbits, "depth": tq.depth(), "gates": dict(tq.count_ops())}

print("Single-cell, theta = pi/3:", resources(bomb_tester(np.pi/3)))

t0 = time.perf_counter(); run(bomb_tester(np.pi/3)); dt = time.perf_counter() - t0
print(f"Run time on Aer, {SHOTS} shots: {dt:.2f} s")

**Qubits and depth.** 2 qubits (path + bomb probe), 2 RX gates and 1 CNOT, depth ≈ 4 (RX, CNOT, RX, measure). 

**Does changing the angle or adding gates help?** Yes. θ = π/2 → 50 %, θ = π/3 → 75 %, θ = π/6 → ≈ 93 % (but detection probability falls to ≈ 6 %). 
Adding more qubits is not needed for a single cell; repeating the probe (quantum Zeno effect) would improve the score further at the cost of extra depth.

**Parameters.** One continuous parameter: the rotation angle θ. There are no trainable variational parameters.

**Run time.** Measured above: well under a few seconds for 10 000 shots on the Aer simulator.

**Noise tolerance.** Moderately tolerant: the circuit is shallow (1 CNOT), so with CNOT error ≲ 1 % the ranking of the angle settings is preserved. Noise adds false alarms and lowers the score as `p` grows.

## 6. Final summary

In [ ]:
rows = []
for name, th in [("Standard (pi/2)", np.pi/2), ("Improved angle (pi/3)", np.pi/3), ("Small angle (pi/6)", np.pi/6)]:
    d, e, i = classify_single(run(bomb_tester(th)))
    rows.append({"setup": name, "detection": d, "explosion": e, "inconclusive": i, "score": score(d, e)})
pd.DataFrame(rows).round(4)